In [16]:
# Cell 1 — Paths for legacy imaging wrangling

from pathlib import Path
import os
import pandas as pd

BASE = Path("/Users/davekokel/Projects/carp_v2")
LEGACY = BASE / "seed_kits" / "legacy_wrangling"
RAW = LEGACY / "raw"
WORKING = LEGACY / "working"

BASE, LEGACY, RAW, WORKING

(PosixPath('/Users/davekokel/Projects/carp_v2'),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling'),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/raw'),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working'))

In [17]:
# Cell 2 — Load canonical ROI annotation table (AUTO)
roi_auto_path = WORKING / "imaging_roi_annotations_AUTO.csv"

if not roi_auto_path.exists():
    raise FileNotFoundError(f"Missing ROI AUTO CSV: {roi_auto_path}")

df_roi = pd.read_csv(roi_auto_path)
df_roi.shape, df_roi.columns

((976, 29),
 Index(['roi_dir', 'date_experiment', 'fish', 'roi_name', 'date_born',
        'parent_female', 'parent_male', 'genotype_base_codes',
        'genotype_allele_codes', 'genotype_pretty',
        'genotype_marker_fluor_codes', 'genotype_marker_tag_codes',
        'treatment_plasmid_base_codes', 'treatment_rna_base_codes',
        'treatment_marker_fluor_codes', 'treatment_marker_tag_codes',
        'all_marker_fluor_codes', 'additional plasmids injected',
        'additional mRNAs injected', 'additonal proteins injected',
        'additonal dye and chemicals', 'Date born', 'ZF female genotype',
        'ZF male genotype', 'Data location', 'plate_id_filled',
        'slot_id_filled', 'roi_index_within_slot', 'roi_code'],
       dtype='object'))

In [18]:
# Cell 3 — Build the grouping key for inferred legacy clutches
# Group by parent_female, parent_male, date_born

group_keys = ["parent_female", "parent_male", "date_born"]

df_roi["legacy_clutch_group"] = (
    df_roi[group_keys]
    .astype(str)
    .agg("|".join, axis=1)
)

df_roi["legacy_clutch_group"].nunique()

61

In [19]:
# Cell 4 — Inferred legacy clutches DataFrame

df_clutches = (
    df_roi.groupby("legacy_clutch_group")
    .agg({
        "parent_female": "first",
        "parent_male": "first",
        "date_born": "first",
        "roi_code": "count",
    })
    .reset_index()
    .rename(columns={
        "roi_code": "roi_count",
    })
)

df_clutches.shape, df_clutches.head()

((61, 5),
                                  legacy_clutch_group  \
 0  Ben (F2 of allele 310)|Abe (F2 of allele 309)|...   
 1  Ben (F2 of allele 310)|Abe (F2 of allele 309)|...   
 2  Dennis (F2 of allele 318)|Abe (F2 of allele 30...   
 3  Dennis (F2 of allele 318)|Abe (F2 of allele 30...   
 4  Dennis (F2 of allele 318)|Abe (F2 of allele 30...   
 
                parent_female             parent_male   date_born  roi_count  
 0     Ben (F2 of allele 310)  Abe (F2 of allele 309)  2025-06-17         23  
 1     Ben (F2 of allele 310)  Abe (F2 of allele 309)  2025-06-24         17  
 2  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-05-20          5  
 3  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-06-04         21  
 4  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-06-11         10  )

In [20]:
# Cell 5 — Assign inferred legacy clutch codes (LCL-####)

df_clutches["clutch_code"] = [
    f"LCL-{i:04d}" for i in range(1, len(df_clutches) + 1)
]

df_clutches = df_clutches[
    ["clutch_code", "parent_female", "parent_male", "date_born", "roi_count", "legacy_clutch_group"]
]

df_clutches.head(), df_clutches.shape

(  clutch_code              parent_female             parent_male   date_born  \
 0    LCL-0001     Ben (F2 of allele 310)  Abe (F2 of allele 309)  2025-06-17   
 1    LCL-0002     Ben (F2 of allele 310)  Abe (F2 of allele 309)  2025-06-24   
 2    LCL-0003  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-05-20   
 3    LCL-0004  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-06-04   
 4    LCL-0005  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-06-11   
 
    roi_count                                legacy_clutch_group  
 0         23  Ben (F2 of allele 310)|Abe (F2 of allele 309)|...  
 1         17  Ben (F2 of allele 310)|Abe (F2 of allele 309)|...  
 2          5  Dennis (F2 of allele 318)|Abe (F2 of allele 30...  
 3         21  Dennis (F2 of allele 318)|Abe (F2 of allele 30...  
 4         10  Dennis (F2 of allele 318)|Abe (F2 of allele 30...  ,
 (61, 6))

In [21]:
# Cell 6 — Attach clutch_code to each ROI row

df_roi_with_clutch = df_roi.merge(
    df_clutches[["legacy_clutch_group", "clutch_code"]],
    on="legacy_clutch_group",
    how="left",
)

df_roi_with_clutch.shape, df_roi_with_clutch[["roi_code", "clutch_code"]].head()

((976, 31),
                       roi_code clutch_code
 0  20250428-plate1-slot1-roi01    LCL-0043
 1  20250428-plate1-slot1-roi02    LCL-0043
 2  20250501-plate1-slot1-roi01    LCL-0053
 3  20250501-plate1-slot1-roi02    LCL-0053
 4  20250501-plate1-slot1-roi03    LCL-0053)

In [22]:
# Cell 7 — Build inferred legacy clutch memberships

df_clutch_memberships = df_roi_with_clutch[[
    "clutch_code",
    "roi_code",
    "plate_id_filled",
    "slot_id_filled",
    "date_born",
    "parent_female",
    "parent_male",
    "roi_dir",
]].copy()

df_clutch_memberships.head(), df_clutch_memberships.shape

(  clutch_code                     roi_code  plate_id_filled  \
 0    LCL-0043  20250428-plate1-slot1-roi01  20250428-plate1   
 1    LCL-0043  20250428-plate1-slot1-roi02  20250428-plate1   
 2    LCL-0053  20250501-plate1-slot1-roi01  20250501-plate1   
 3    LCL-0053  20250501-plate1-slot1-roi02  20250501-plate1   
 4    LCL-0053  20250501-plate1-slot1-roi03  20250501-plate1   
 
           slot_id_filled   date_born                            parent_female  \
 0  20250428-plate1-slot1  2025-04-26  ef1a:2xLynk:tdmSG(J) (F2 of allele 301)   
 1  20250428-plate1-slot1  2025-04-26  ef1a:2xLynk:tdmSG(J) (F2 of allele 301)   
 2  20250501-plate1-slot1  2025-04-29                            membrane Halo   
 3  20250501-plate1-slot1  2025-04-29                            membrane Halo   
 4  20250501-plate1-slot1  2025-04-29                            membrane Halo   
 
                                parent_male  \
 0  ef1a:2xLynk:tdmSG(J) (F2 of allele 301)   
 1  ef1a:2xLynk:tdmSG(J) (

In [23]:
# Cell 8 — Write inferred clutches + memberships to working directory

clutches_out = WORKING / "legacy_clutches.csv"
memberships_out = WORKING / "legacy_clutch_memberships.csv"

df_clutches.to_csv(clutches_out, index=False)
df_clutch_memberships.to_csv(memberships_out, index=False)

clutches_out, memberships_out

(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/legacy_clutches.csv'),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/legacy_clutch_memberships.csv'))

In [24]:
df_roi = pd.read_csv(roi_auto_path)
df_clutches  # built from grouping

,clutch_code,parent_female,parent_male,date_born,roi_count,legacy_clutch_group
0,LCL-0001,Ben (F2 of allele 310),Abe (F2 of allele 309),2025-06-17,23,Ben (F2 of allele 310)|Abe (F2 of allele 309)|...
1,LCL-0002,Ben (F2 of allele 310),Abe (F2 of allele 309),2025-06-24,17,Ben (F2 of allele 310)|Abe (F2 of allele 309)|...
2,LCL-0003,Dennis (F2 of allele 318),Abe (F2 of allele 309),2025-05-20,5,Dennis (F2 of allele 318)|Abe (F2 of allele 30...
3,LCL-0004,Dennis (F2 of allele 318),Abe (F2 of allele 309),2025-06-04,21,Dennis (F2 of allele 318)|Abe (F2 of allele 30...
4,LCL-0005,Dennis (F2 of allele 318),Abe (F2 of allele 309),2025-06-11,10,Dennis (F2 of allele 318)|Abe (F2 of allele 30...
...,...,...,...,...,...,...
56,LCL-0057,membrane mChilada,membrane mChilada,2025-08-07,21,membrane mChilada|membrane mChilada|2025-08-07
57,LCL-0058,None,ef1a:2xLynk:tdm:Halo,2025-09-12,18,nan|ef1a:2xLynk:tdm:Halo|2025-09-12
58,LCL-0059,None,None,2025-05-21,19,nan|nan|2025-05-21
59,LCL-0060,None,None,2025-05-27,2,nan|nan|2025-05-27


In [25]:
# How many unique parent pairs?
df_roi[["parent_female", "parent_male"]].drop_duplicates().shape

(26, 2)

In [26]:
# How many distinct dates of birth in the ROI table?
df_roi["date_born"].dropna().nunique()

51

In [27]:
# Find the group key for the big unknown clutch (LCL-0061)
big_unknown_group = df_clutches.loc[df_clutches["clutch_code"] == "LCL-0061", "legacy_clutch_group"].iloc[0]
big_unknown_group

'nan|nan|nan'

In [29]:
# Rebuild the grouping key for inferred legacy clutches in this session
group_keys = ["parent_female", "parent_male", "date_born"]

df_roi["legacy_clutch_group"] = (
    df_roi[group_keys]
    .astype(str)
    .agg("|".join, axis=1)
)

df_roi["legacy_clutch_group"].nunique()

61

In [33]:
big_unknown_group = df_clutches.loc[
    df_clutches["clutch_code"] == "LCL-0061",
    "legacy_clutch_group"
].iloc[0]
# returns 'nan|nan|nan'
df_unknown = df_roi[df_roi["legacy_clutch_group"] == big_unknown_group].copy()
df_unknown.shape
df_unknown[[
    "date_experiment",
    "date_born",
    "parent_female",
    "parent_male",
    "roi_name",
    "roi_dir",
    "Data location",
]].head(20)

,date_experiment,date_born,parent_female,parent_male,roi_name,roi_dir,Data location
825,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,er_roi1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN
826,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,er_roi2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN
827,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,er_roi3,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN
828,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,mito_roi1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN
829,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,NaN,NaN,NaN,mito_roi2,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN
830,20250723_er-mSG_mem-mChilada,NaN,NaN,NaN,fish1_24hpf_roi1,/clusterfs/vast/abcabc/Aang_Foundation/2025072...,NaN
831,20251002_mem_mito,NaN,NaN,NaN,fish10_24hpf_roi1,/clusterfs/vast/abcabc/Aang_Foundation/2025100...,NaN
832,20251002_mem_mito,NaN,NaN,NaN,fish10_24hpf_roi2,/clusterfs/vast/abcabc/Aang_Foundation/2025100...,NaN
833,20251002_mem_mito,NaN,NaN,NaN,fish11_24hpf_roi1,/clusterfs/vast/abcabc/Aang_Foundation/2025100...,NaN
834,20251002_mem_mito,NaN,NaN,NaN,fish11_24hpf_roi2,/clusterfs/vast/abcabc/Aang_Foundation/2025100...,NaN


In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell A: rebuild legacy_clutch_group from current df_roi
# ─────────────────────────────────────────────────────────────────────────────

import re
import pandas as pd

# group by parent_female, parent_male, date_born (as before)
group_keys = ["parent_female", "parent_male", "date_born"]

df_roi["legacy_clutch_group"] = (
    df_roi[group_keys]
    .astype(str)
    .agg("|".join, axis=1)
)

df_roi["legacy_clutch_group"].nunique()

61

In [35]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell B: identify the fully-unknown group (nan|nan|nan)
# ─────────────────────────────────────────────────────────────────────────────

# find groups where all three keys are actually NaN in the raw columns
mask_unknown = (
    df_roi["parent_female"].isna()
    & df_roi["parent_male"].isna()
    & df_roi["date_born"].isna()
)

# the corresponding legacy_clutch_group label(s)
unknown_groups = df_roi.loc[mask_unknown, "legacy_clutch_group"].unique()
unknown_groups

array(['nan|nan|nan'], dtype=object)

In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell C: extract imaging_date and hpf for the unknown group rows
# ─────────────────────────────────────────────────────────────────────────────

# subset of the ROI table that is in an unknown group
df_unknown = df_roi[df_roi["legacy_clutch_group"].isin(unknown_groups)].copy()
df_unknown.shape

# helper: extract an 8-digit date from a string (e.g. "20250521_mem-histone")
def extract_imaging_date(s: str):
    if not isinstance(s, str):
        return pd.NaT
    m = re.search(r"(\d{8})", s)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y%m%d").date()
        except Exception:
            return pd.NaT
    return pd.NaT

# helper: extract hpf from text like "24hpf", "24 hpf"
def extract_hpf(s: str):
    if not isinstance(s, str):
        return None
    m = re.search(r"(\d+)\s*hpf", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"(\d+)hpf", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"(\d+)\s*h", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None

# imaging_date: try date_experiment first, then roi_dir
df_unknown["imaging_date"] = df_unknown["date_experiment"].apply(extract_imaging_date)
df_unknown.loc[df_unknown["imaging_date"].isna(), "imaging_date"] = (
    df_unknown.loc[df_unknown["imaging_date"].isna(), "roi_dir"].apply(extract_imaging_date)
)

# hpf: try roi_name, then fish
df_unknown["hpf"] = df_unknown["roi_name"].apply(extract_hpf)
df_unknown.loc[df_unknown["hpf"].isna(), "hpf"] = df_unknown["fish"].apply(extract_hpf)

df_unknown[["date_experiment", "roi_name", "fish", "imaging_date", "hpf"]].head(20)

,date_experiment,roi_name,fish,imaging_date,hpf
825,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,er_roi1,er_roi1,2025-07-21,NaN
826,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,er_roi2,er_roi2,2025-07-21,NaN
827,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,er_roi3,er_roi3,2025-07-21,NaN
828,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,mito_roi1,mito_roi1,2025-07-21,NaN
829,20250721_72hpf_mrna_mSG_organelle_LLS-SIM,mito_roi2,mito_roi2,2025-07-21,NaN
830,20250723_er-mSG_mem-mChilada,fish1_24hpf_roi1,fish1_24hpf_roi1,2025-07-23,24.0
831,20251002_mem_mito,fish10_24hpf_roi1,fish10_24hpf_roi1,2025-10-02,24.0
832,20251002_mem_mito,fish10_24hpf_roi2,fish10_24hpf_roi2,2025-10-02,24.0
833,20251002_mem_mito,fish11_24hpf_roi1,fish11_24hpf_roi1,2025-10-02,24.0
834,20251002_mem_mito,fish11_24hpf_roi2,fish11_24hpf_roi2,2025-10-02,24.0


In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell D: infer birthdate as imaging_date - (hpf/24 days)
# ─────────────────────────────────────────────────────────────────────────────

# hpf may be None; fill with 0 so we don't explode
df_unknown["hpf_filled"] = df_unknown["hpf"].fillna(0)

df_unknown["inferred_birthdate"] = (
    df_unknown["imaging_date"]
    - pd.to_timedelta(df_unknown["hpf_filled"] / 24, unit="D")
)

df_unknown[["roi_code", "imaging_date", "hpf", "inferred_birthdate"]].head(20)
df_unknown["inferred_birthdate"].value_counts()

/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_95491/1927136001.py:9: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_unknown["imaging_date"]


inferred_birthdate
2025-05-13    44
2025-10-01    31
2025-10-05    25
2025-06-27    12
2025-06-25    10
2025-10-29     8
2025-07-21     5
2025-10-07     4
2025-07-23     4
2025-10-28     2
2025-07-22     1
2025-11-04     1
2025-06-20     1
2025-08-01     1
2025-10-09     1
Name: count, dtype: int64

In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell E: write inferred_birthdate back into df_roi for unknown-group rows
# ─────────────────────────────────────────────────────────────────────────────

# we'll only overwrite date_born where it was originally NaN
mask_unknown_global = df_roi["legacy_clutch_group"].isin(unknown_groups)

# align df_unknown rows back to df_roi by roi_code (safer than relying on index)
df_unknown_birth = df_unknown[["roi_code", "inferred_birthdate"]].set_index("roi_code")

# attach inferred_birthdate to df_roi via roi_code
df_roi = df_roi.set_index("roi_code")
df_roi.loc[df_unknown_birth.index, "date_born"] = df_unknown_birth["inferred_birthdate"]
df_roi = df_roi.reset_index()  # roi_code back to column

df_roi["date_born"].isna().sum(), df_roi.shape

(np.int64(10), (976, 30))

In [39]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell F: rebuild legacy_clutch_group and clutches with updated date_born
# ─────────────────────────────────────────────────────────────────────────────

group_keys = ["parent_female", "parent_male", "date_born"]

df_roi["legacy_clutch_group"] = (
    df_roi[group_keys]
    .astype(str)
    .agg("|".join, axis=1)
)

n_groups = df_roi["legacy_clutch_group"].nunique()
n_groups

81

In [40]:
# rebuild df_clutches with the updated grouping
df_clutches = (
    df_roi.groupby("legacy_clutch_group")
    .agg({
        "parent_female": "first",
        "parent_male": "first",
        "date_born": "first",
        "roi_code": "count",
    })
    .reset_index()
    .rename(columns={"roi_code": "roi_count"})
)

# assign new clutch_code: you can keep LCL-#### or change the pattern later
df_clutches["clutch_code"] = [
    f"LCL-{i:04d}" for i in range(1, len(df_clutches) + 1)
]

df_clutches = df_clutches[
    ["clutch_code", "parent_female", "parent_male", "date_born", "roi_count", "legacy_clutch_group"]
]

df_clutches.shape, df_clutches.head()

((81, 6),
   clutch_code              parent_female             parent_male   date_born  \
 0    LCL-0001     Ben (F2 of allele 310)  Abe (F2 of allele 309)  2025-06-17   
 1    LCL-0002     Ben (F2 of allele 310)  Abe (F2 of allele 309)  2025-06-24   
 2    LCL-0003  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-05-20   
 3    LCL-0004  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-06-04   
 4    LCL-0005  Dennis (F2 of allele 318)  Abe (F2 of allele 309)  2025-06-11   
 
    roi_count                                legacy_clutch_group  
 0         23  Ben (F2 of allele 310)|Abe (F2 of allele 309)|...  
 1         17  Ben (F2 of allele 310)|Abe (F2 of allele 309)|...  
 2          5  Dennis (F2 of allele 318)|Abe (F2 of allele 30...  
 3         21  Dennis (F2 of allele 318)|Abe (F2 of allele 30...  
 4         10  Dennis (F2 of allele 318)|Abe (F2 of allele 30...  )

In [41]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell G: reattach clutch_code to each ROI row and rebuild memberships
# ─────────────────────────────────────────────────────────────────────────────

df_roi_with_clutch = df_roi.merge(
    df_clutches[["legacy_clutch_group", "clutch_code"]],
    on="legacy_clutch_group",
    how="left",
)

df_roi_with_clutch.shape, df_roi_with_clutch[["roi_code", "clutch_code"]].head()

df_clutch_memberships = df_roi_with_clutch[[
    "clutch_code",
    "roi_code",
    "plate_id_filled",
    "slot_id_filled",
    "date_born",
    "parent_female",
    "parent_male",
    "roi_dir",
]].copy()

df_clutch_memberships.shape, df_clutch_memberships.head()

((976, 8),
   clutch_code                     roi_code  plate_id_filled  \
 0    LCL-0047  20250428-plate1-slot1-roi01  20250428-plate1   
 1    LCL-0047  20250428-plate1-slot1-roi02  20250428-plate1   
 2    LCL-0057  20250501-plate1-slot1-roi01  20250501-plate1   
 3    LCL-0057  20250501-plate1-slot1-roi02  20250501-plate1   
 4    LCL-0057  20250501-plate1-slot1-roi03  20250501-plate1   
 
           slot_id_filled   date_born                            parent_female  \
 0  20250428-plate1-slot1  2025-04-26  ef1a:2xLynk:tdmSG(J) (F2 of allele 301)   
 1  20250428-plate1-slot1  2025-04-26  ef1a:2xLynk:tdmSG(J) (F2 of allele 301)   
 2  20250501-plate1-slot1  2025-04-29                            membrane Halo   
 3  20250501-plate1-slot1  2025-04-29                            membrane Halo   
 4  20250501-plate1-slot1  2025-04-29                            membrane Halo   
 
                                parent_male  \
 0  ef1a:2xLynk:tdmSG(J) (F2 of allele 301)   
 1  ef1a:2xLynk

In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell H: write updated clutches + memberships to CSVs
# ─────────────────────────────────────────────────────────────────────────────

clutches_out = WORKING / "legacy_clutches.csv"
memberships_out = WORKING / "legacy_clutch_memberships.csv"

df_clutches.to_csv(clutches_out, index=False)
df_clutch_memberships.to_csv(memberships_out, index=False)

clutches_out, memberships_out

(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/legacy_clutches.csv'),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/legacy_clutch_memberships.csv'))

In [43]:
df_roi["date_born"].isna().sum()

np.int64(10)

In [44]:
df_roi["date_born"].isna().mean() * 100

np.float64(1.0245901639344261)

In [45]:
df_roi[df_roi["date_born"].isna()].head()


,roi_code,roi_dir,date_experiment,fish,roi_name,date_born,parent_female,parent_male,genotype_base_codes,genotype_allele_codes,...,additonal proteins injected,additonal dye and chemicals,Date born,ZF female genotype,ZF male genotype,Data location,plate_id_filled,slot_id_filled,roi_index_within_slot,legacy_clutch_group
10,20250520-plate1-slot3-roi01,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250520_mem_histone,fish2,roi1,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",...,NaN,NaN,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250520_mem_histon...,20250520-plate1,20250520-plate1-slot3,1.0,ef1a:2xLynk:tdmSG(J) (F2 of allele 301)|ef1a:2...
11,20250520-plate1-slot3-roi02,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250520_mem_histone,fish4,roi1_tailbud,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",...,NaN,NaN,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250520_mem_histon...,20250520-plate1,20250520-plate1-slot3,2.0,ef1a:2xLynk:tdmSG(J) (F2 of allele 301)|ef1a:2...
12,20250520-plate1-slot3-roi03,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250520_mem_histone,fish4,roi1_tailbud_2,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",...,NaN,NaN,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250520_mem_histon...,20250520-plate1,20250520-plate1-slot3,3.0,ef1a:2xLynk:tdmSG(J) (F2 of allele 301)|ef1a:2...
13,20250520-plate1-slot3-roi04,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250520_mem_histone,fish5,roi1_tailbud,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",...,NaN,NaN,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250520_mem_histon...,20250520-plate1,20250520-plate1-slot3,4.0,ef1a:2xLynk:tdmSG(J) (F2 of allele 301)|ef1a:2...
38,20250523-plate1-slot1-roi01,/clusterfs/vast/abcabc/Korra_Foundation/202505...,20250523_mem_histone,fish1,roi1,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),pDQM005,"pDQM005:302,pDQM005:302",...,NaN,NaN,NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),X:\abcabc\Korra_Foundation\20250523_mem_histon...,20250523-plate1,20250523-plate1-slot1,1.0,ef1a:2xLynk:tdmSG(J) (F2 of allele 302)|ef1a:2...


In [46]:
# Cell: infer missing birthdays using sibling clutches (same parents)

df_roi["date_born_fixed"] = df_roi["date_born"]

# parent pair grouping
parent_groups = df_roi.groupby(["parent_female", "parent_male"])

for (pf, pm), group in parent_groups:
    # birthday values present in siblings
    known_births = group["date_born"].dropna().unique()
    if len(known_births) == 1:
        inferred_birth = known_births[0]
        # fill only missing values in this group
        mask = (
            (df_roi["parent_female"] == pf) &
            (df_roi["parent_male"] == pm) &
            (df_roi["date_born"].isna())
        )
        df_roi.loc[mask, "date_born_fixed"] = inferred_birth

df_roi["date_born"] = df_roi["date_born_fixed"]
df_roi = df_roi.drop(columns=["date_born_fixed"])

df_roi["date_born"].isna().sum(), df_roi["date_born"].nunique()

(np.int64(10), 66)

In [47]:
missing = df_roi[df_roi["date_born"].isna()]
missing[[
    "roi_code", "date_experiment", "roi_name", 
    "parent_female", "parent_male", "plate_id_filled"
]]

,roi_code,date_experiment,roi_name,parent_female,parent_male,plate_id_filled
10,20250520-plate1-slot3-roi01,20250520_mem_histone,roi1,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),20250520-plate1
11,20250520-plate1-slot3-roi02,20250520_mem_histone,roi1_tailbud,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),20250520-plate1
12,20250520-plate1-slot3-roi03,20250520_mem_histone,roi1_tailbud_2,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),20250520-plate1
13,20250520-plate1-slot3-roi04,20250520_mem_histone,roi1_tailbud,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),20250520-plate1
38,20250523-plate1-slot1-roi01,20250523_mem_histone,roi1,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),20250523-plate1
39,20250523-plate1-slot1-roi02,20250523_mem_histone,roi1,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),20250523-plate1
40,20250523-plate1-slot1-roi03,20250523_mem_histone,roi1,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),20250523-plate1
41,20250523-plate1-slot1-roi04,20250523_mem_histone,roi1,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),20250523-plate1
42,20250523-plate1-slot1-roi05,20250523_mem_histone,roi1,ef1a:2xLynk:tdmSG(J) (F2 of allele 302),ef1a:2xLynk:tdmSG(J) (F2 of allele 302),20250523-plate1
975,NaN,analysis_test,fish1_roi1,NaN,NaN,NaN


In [48]:
for idx, row in missing.iterrows():
    pf = row["parent_female"]
    pm = row["parent_male"]
    
    print("\n=== MISSING ROI ===")
    print(row[["roi_code", "date_experiment", "roi_name"]])
    print("Parents:", pf, "|", pm)
    
    sibs = df_roi[
        (df_roi["parent_female"] == pf) &
        (df_roi["parent_male"] == pm) &
        (df_roi["date_born"].notna())
    ]
    
    print(f"Siblings with known birthday: {len(sibs)}")
    if len(sibs):
        print(sibs[["roi_code","date_born","date_experiment"]].head())
    else:
        print("⚠️ NO siblings with known birthdays")


=== MISSING ROI ===
roi_code           20250520-plate1-slot3-roi01
date_experiment           20250520_mem_histone
roi_name                                  roi1
Name: 10, dtype: object
Parents: ef1a:2xLynk:tdmSG(J) (F2 of allele 301) | ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
Siblings with known birthday: 43
                        roi_code   date_born       date_experiment
0    20250428-plate1-slot1-roi01  2025-04-26  20250428_mem_histone
1    20250428-plate1-slot1-roi02  2025-04-26  20250428_mem_histone
136  20250610-plate1-slot1-roi01  2025-06-09  20250609_mem-histone
137  20250610-plate1-slot1-roi02  2025-06-09  20250609_mem-histone
138  20250610-plate1-slot1-roi03  2025-06-09  20250609_mem-histone

=== MISSING ROI ===
roi_code           20250520-plate1-slot3-roi02
date_experiment           20250520_mem_histone
roi_name                          roi1_tailbud
Name: 11, dtype: object
Parents: ef1a:2xLynk:tdmSG(J) (F2 of allele 301) | ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
Siblings wi

In [49]:
# === Fill missing birthdays using parent-pair sibling inference ===

df_roi["date_born_fixed"] = df_roi["date_born"].copy()

# group by parent pairs
g = df_roi.groupby(["parent_female", "parent_male"])

for (pf, pm), grp in g:
    known_dates = grp["date_born"].dropna().unique()
    
    if len(known_dates) == 1:
        inferred = known_dates[0]

        mask = (
            (df_roi["parent_female"] == pf) &
            (df_roi["parent_male"] == pm) &
            (df_roi["date_born"].isna())
        )
        
        df_roi.loc[mask, "date_born_fixed"] = inferred

df_roi["date_born"] = df_roi["date_born_fixed"]
df_roi = df_roi.drop(columns=["date_born_fixed"])

# re-check count
df_roi["date_born"].isna().sum()

np.int64(10)

In [51]:
# === FIX ALL REMAINING MISSING BIRTHDAYS ===

# 1. Drop the garbage analysis_test row
df_roi = df_roi[~df_roi["date_experiment"].eq("analysis_test")].copy()

# 2. Apply sibling inference again cleanly

df_roi["date_born_fixed"] = df_roi["date_born"]

groups = df_roi.groupby(["parent_female", "parent_male"])

for (pf, pm), grp in groups:
    known = grp["date_born"].dropna().unique()
    if len(known) == 1:
        inferred = known[0]
        mask = (
            (df_roi["parent_female"] == pf) &
            (df_roi["parent_male"] == pm) &
            (df_roi["date_born"].isna())
        )
        df_roi.loc[mask, "date_born_fixed"] = inferred

df_roi["date_born"] = df_roi["date_born_fixed"]
df_roi = df_roi.drop(columns=["date_born_fixed"])

df_roi["date_born"].isna().sum()

np.int64(9)

In [52]:
# Cell X: Fix the last missing birthdays pragmatically (assume 24hpf) and drop analysis_test

import pandas as pd
import re

# 1. Drop the obvious garbage row from analysis_test
df_roi = df_roi[~df_roi["date_experiment"].eq("analysis_test")].copy()

# 2. Identify rows with missing date_born but real parents (the 9 ROIs we care about)
missing = df_roi[
    df_roi["date_born"].isna()
    & df_roi["parent_female"].notna()
    & df_roi["parent_male"].notna()
].copy()

missing_idx = missing.index
print("Missing with real parents:", len(missing_idx))

# 3. Extract imaging_date from date_experiment (YYYYMMDD prefix)
def extract_imaging_date(s: str):
    if not isinstance(s, str):
        return pd.NaT
    m = re.search(r"(\d{8})", s)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y%m%d").date()
        except Exception:
            return pd.NaT
    return pd.NaT

missing["imaging_date"] = missing["date_experiment"].apply(extract_imaging_date)

# 4. Assume 24hpf: birthday = imaging_date - 1 day
missing["date_born_inferred_24hpf"] = missing["imaging_date"] - pd.to_timedelta(1, unit="D")

# 5. Write inferred birthdays back into df_roi.date_born
for idx, row in missing.iterrows():
    df_roi.loc[idx, "date_born"] = row["date_born_inferred_24hpf"]

df_roi["date_born"].isna().sum(), df_roi["date_born"].nunique()

Missing with real parents: 9


(np.int64(0), 68)